# Stage 1 Autoencoder Ablation: Multi-Source ALE Maps

This notebook runs only the Stage 1 autoencoder experiments. It compares mixed-source Stage 1A AE recipes and optional Stage 1B domain fine-tuning for PubMed, Nilearn, and NeuroVault before any downstream contrastive or generation runs.

Use this notebook to produce candidate AE checkpoints and reconstruction-quality summaries. Downstream checkpoint selection happens in notebook 5b.


In [ ]:
from pathlib import Path
import csv
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.training.train_autoencoder import evaluate_saved_checkpoints_from_config, train_from_config, train_stage1b_from_config
from atlas_free_cnn.pipeline_outputs import AE_SELECTION_TO_FILE, create_full_pipeline_run_dir, write_table


In [ ]:
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    from huggingface_hub import hf_hub_download
    local_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for filename in filenames:
        try:
            path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=filename,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
            )
            return Path(path)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise FileNotFoundError("Could not download any candidate from HF:\n" + "\n".join(errors))


def ensure_hf_unified_splits() -> Path:
    print(f"Downloading atlas-free CNN split JSONLs from Hugging Face: {HF_DATASET_REPO}")
    LOCAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "val", "test"]:
        downloaded = hf_download_first_available(
            [f"splits/{split}.jsonl", f"unified_jsonl_rebuild/splits/{split}.jsonl", f"{split}.jsonl"],
            LOCAL_UNIFIED_CACHE_DIR,
        )
        target = LOCAL_SPLIT_DIR / f"{split}.jsonl"
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
    for name in ["train_map_ids.json", "val_map_ids.json", "test_map_ids.json"]:
        try:
            downloaded = hf_download_first_available(
                [f"splits/{name}", f"unified_jsonl_rebuild/splits/{name}", name],
                LOCAL_UNIFIED_CACHE_DIR,
            )
            target = LOCAL_SPLIT_DIR / name
            if downloaded.resolve() != target.resolve():
                shutil.copy2(downloaded, target)
        except Exception as exc:
            print(f"Optional split sidecar not downloaded ({name}): {exc}")
    try:
        downloaded_volume = hf_download_first_available(
            ["atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn/atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn_rebuild/atlas_free_cnn_volumes.pt"],
            LOCAL_PACK_DIR,
        )
        target_volume = LOCAL_PACK_DIR / "atlas_free_cnn_volumes.pt"
        if downloaded_volume.resolve() != target_volume.resolve():
            try:
                if target_volume.exists() or target_volume.is_symlink():
                    target_volume.unlink()
                os.symlink(downloaded_volume, target_volume)
            except Exception:
                shutil.copy2(downloaded_volume, target_volume)
        print("Volume tensor available at:", target_volume)
    except Exception as exc:
        print("WARNING: split JSONLs downloaded, but volume tensor was not prepared:", exc)
        print("Training will fail unless tensor_path values inside JSONL resolve to an accessible tensor file.")
    return LOCAL_SPLIT_DIR


def discover_unified_split_dir() -> Path:
    from atlas_free_cnn.notebook_utils import discover_unified_split_dir as _discover_unified_split_dir
    return _discover_unified_split_dir(
        repo_dir=REPO_DIR,
        drive_root=DRIVE_ROOT,
        dataset_repo=HF_DATASET_REPO,
        local_unified_cache_dir=LOCAL_UNIFIED_CACHE_DIR,
        local_split_dir=LOCAL_SPLIT_DIR,
        local_pack_dir=LOCAL_PACK_DIR,
    )
UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

# quick: short smoke/iteration run; full: real AE ablation run.
AE_ABLATION_MODE = "full"  # quick | full
TRAIN_OTHER_AE_VARIANTS = False  # True also trains balanced raw MSE and balanced hybrid-loss AE recipes
SELECTED_AE_VARIANTS = "all" if TRAIN_OTHER_AE_VARIANTS else ["mixed_baseline_raw_mse"]
AE_CHECKPOINT_SELECTION = "best_top5_dice"
RUN_STAGE1A_MIXED_PRETRAINING = True
RUN_STAGE1B_FINETUNING = False
RUN_STAGE1B = RUN_STAGE1B_FINETUNING  # compatibility alias for older saved outputs
RESUME_COMPLETED = True
LOAD_STAGE1A = {"enabled": False, "source": "hf", "run_dir": os.environ.get("NEUROVLM_STAGE1A_SOURCE_RUN_DIR", "")}
SAVE_LEGACY_AE_CHECKPOINT_ALIASES = False  # write best_cnn_autoencoder.pt / last_cnn_autoencoder.pt for backward compat with older runs
STAGE1B_SEED_VARIANT = "mixed_baseline_raw_mse"
STAGE1B_DOMAINS = [
    ("mixed_pretrain_to_pubmed", "pubmed", "mixed_baseline_to_pubmed"),
    ("mixed_pretrain_to_neurovault", "neurovault", "mixed_baseline_to_neurovault"),
    ("mixed_pretrain_to_nilearn", "nilearn", "mixed_baseline_to_nilearn"),
]

AE_EPOCHS = 2 if AE_ABLATION_MODE == "quick" else 300
STAGE1B_EPOCHS = 2 if AE_ABLATION_MODE == "quick" else 100
AE_EARLY_STOPPING_PATIENCE = 15
MAX_TRAIN_BATCHES = 20 if AE_ABLATION_MODE == "quick" else None
MAX_VAL_BATCHES = 5 if AE_ABLATION_MODE == "quick" else None
TRAIN_METRIC_BATCHES = 2 if AE_ABLATION_MODE == "quick" else 0
VAL_METRIC_BATCHES = 2 if AE_ABLATION_MODE == "quick" else 16
AE_BATCH_CANDIDATES = [512, 384, 256, 192, 128, 96, 64, 48, 32, 16]
AE_PREFLIGHT_RESERVE_GB = 18.0 if AE_ABLATION_MODE == "quick" else 12.0
STAGE1B_BATCH_CANDIDATES = [96, 64, 48, 32, 16]
STAGE1B_PREFLIGHT_RESERVE_GB = 28.0
NUM_WORKERS = int(os.environ.get("NEUROVLM_NUM_WORKERS", "4" if IN_COLAB else "0"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", str(NUM_WORKERS)))
PREFETCH_FACTOR = int(os.environ.get("NEUROVLM_PREFETCH_FACTOR", "4"))
METRICS_DEVICE = os.environ.get("NEUROVLM_METRICS_DEVICE", "cuda")
COMPUTE_EPOCH_SOURCE_METRICS = os.environ.get("NEUROVLM_EPOCH_SOURCE_METRICS", "0") == "1"
COMPUTE_TRAIN_METRICS = AE_ABLATION_MODE == "quick" or os.environ.get("NEUROVLM_TRAIN_METRICS", "0") == "1"
FINAL_EVAL = AE_ABLATION_MODE == "full"
SAVE_PLOTS = AE_ABLATION_MODE == "full"

ABLATION_OUTPUT_DIR = DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation" if "DRIVE_ROOT" in globals() else Path("runs_atlas_free_cnn_ae_ablation")
paths = create_full_pipeline_run_dir(ABLATION_OUTPUT_DIR, prefix="ae_ablation")
RUN_DIR = Path(paths["run_dir"])
print("Run directory:", RUN_DIR)
print({
    "AE_ABLATION_MODE": AE_ABLATION_MODE,
    "AE_EPOCHS": AE_EPOCHS,
    "AE_EARLY_STOPPING_PATIENCE": AE_EARLY_STOPPING_PATIENCE,
    "TRAIN_OTHER_AE_VARIANTS": TRAIN_OTHER_AE_VARIANTS,
    "SELECTED_AE_VARIANTS": SELECTED_AE_VARIANTS,
    "MAX_TRAIN_BATCHES": MAX_TRAIN_BATCHES,
    "MAX_VAL_BATCHES": MAX_VAL_BATCHES,
    "AE_BATCH_CANDIDATES": AE_BATCH_CANDIDATES,
    "STAGE1B_BATCH_CANDIDATES": STAGE1B_BATCH_CANDIDATES,
    "RUN_STAGE1A_MIXED_PRETRAINING": RUN_STAGE1A_MIXED_PRETRAINING,
    "RUN_STAGE1B_FINETUNING": RUN_STAGE1B_FINETUNING,
    "LOAD_STAGE1A": LOAD_STAGE1A,
    "STAGE1B_SEED_VARIANT": STAGE1B_SEED_VARIANT,
    "STAGE1B_DOMAINS": STAGE1B_DOMAINS,
    "NUM_WORKERS": NUM_WORKERS,
    "METRICS_DEVICE": METRICS_DEVICE,
    "COMPUTE_TRAIN_METRICS": COMPUTE_TRAIN_METRICS,
    "AE_PREFLIGHT_RESERVE_GB": AE_PREFLIGHT_RESERVE_GB,
    "FINAL_EVAL": FINAL_EVAL,
    "SAVE_PLOTS": SAVE_PLOTS,
    "RUN_STAGE1B": RUN_STAGE1B,
})


In [ ]:
recipes = [
    {
        "ae_variant": "mixed_baseline_raw_mse",
        "ae_training_recipe": "baseline_raw_mse",
        "source_sampling": "natural",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_val_loss",
    },
    {
        "ae_variant": "mixed_balanced_raw_mse",
        "ae_training_recipe": "mixed_balanced_raw_mse",
        "source_sampling": "balanced",
        "loss": {"type": "raw_mse", "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
    {
        "ae_variant": "mixed_balanced_hybrid_loss",
        "ae_training_recipe": "mixed_balanced_hybrid_loss",
        "source_sampling": "balanced",
        "loss": {"type": "hybrid_recon", "lambda_foreground": 0.10, "lambda_topk": 0.05, "topk_percent": 5, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
]

In [ ]:
import gc
import torch

results = []
stage1a_status = "not_started"
stage1a_trained_count = 0
stage1a_loaded_existing_count = 0
selected = {r["ae_variant"]: r for r in recipes}
if SELECTED_AE_VARIANTS == "all":
    run_recipes = recipes
else:
    run_recipes = [selected[name] for name in SELECTED_AE_VARIANTS]


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def is_cuda_oom(exc: BaseException) -> bool:
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "cuda out of memory" in text or "outofmemoryerror" in text


def train_with_batch_fallback(base_cfg, candidates, *, reserve_gb=AE_PREFLIGHT_RESERVE_GB, train_fn=train_from_config):
    candidates = [int(v) for v in candidates]
    last_exc = None
    for max_batch in candidates:
        clear_cuda_memory()
        cfg = dict(base_cfg)
        cfg["batch_candidates"] = [v for v in candidates if v <= max_batch]
        cfg["max_batch_size"] = max_batch
        cfg["batch_size"] = min(int(cfg.get("batch_size", 64)), max_batch)
        cfg["preflight_vram_reserve_gb"] = reserve_gb
        print(f"Trying max AE batch size {max_batch}; candidates={cfg['batch_candidates']}")
        try:
            return train_fn(cfg)
        except Exception as exc:
            if not is_cuda_oom(exc):
                raise
            last_exc = exc
            print(f"OOM at max_batch_size={max_batch}. Falling back to next smaller candidate.")
            clear_cuda_memory()
    raise RuntimeError(f"All AE batch candidates failed: {candidates}") from last_exc


def ensure_checkpoint_eval_table(run_dir):
    run_dir = Path(run_dir)
    checkpoint_metrics_path = run_dir / "metrics" / "reconstruction_summary_by_checkpoint_source.csv"
    if checkpoint_metrics_path.exists():
        return checkpoint_metrics_path
    config_path = run_dir / "autoencoder_config.json"
    if not config_path.exists():
        config_path = run_dir / "config" / "ae_config.json"
    if not config_path.exists():
        print(f"WARNING: cannot evaluate checkpoints because config is missing under {run_dir}")
        return checkpoint_metrics_path
    print(f"Evaluating saved checkpoints for {run_dir}")
    evaluate_saved_checkpoints_from_config(config_path)
    return checkpoint_metrics_path


def load_stage1a_results_from_run(source_run_dir):
    source_run_dir = Path(source_run_dir).expanduser()
    if not source_run_dir.exists():
        raise FileNotFoundError(f"Stage 1A source run dir does not exist: {source_run_dir}")
    loaded = []
    for recipe in run_recipes:
        variant = recipe["ae_variant"]
        out_dir = source_run_dir / "01_stage1_ae_pretraining" / variant
        ckpt_dir = out_dir / "checkpoints"
        sel_metric = recipe.get("checkpoint_selection_metric", "best_val_loss")
        canonical_name = AE_SELECTION_TO_FILE.get(sel_metric, "best_val_loss.pt")
        best_checkpoint = ckpt_dir / canonical_name
        if not best_checkpoint.exists():
            raise FileNotFoundError(f"Missing completed Stage 1A checkpoint for {variant}: {best_checkpoint}")
        done_marker = out_dir / "training_stop.json"
        if not done_marker.exists():
            print(f"WARNING: Stage 1A done marker is missing for {variant}: {done_marker}")
        ensure_checkpoint_eval_table(out_dir)
        print(f"Loaded completed Stage 1A variant {variant}: {best_checkpoint}")
        loaded.append({"ae_variant": variant, "checkpoint_dir": str(ckpt_dir), "best_checkpoint": str(best_checkpoint)})
    return loaded


def load_stage1a_result_from_hf():
    from neurovlm.retrieval_resources import _download_from_hf

    checkpoint_path = Path(_download_from_hf("neurovlm/3d_cnn", "mixed_ae.pt", repo_type="model"))
    print("Using Hugging Face mixed Stage 1A checkpoint for Stage 1B fine-tuning:", checkpoint_path)
    return [{
        "ae_variant": STAGE1B_SEED_VARIANT,
        "checkpoint_dir": str(checkpoint_path.parent),
        "best_checkpoint": str(checkpoint_path),
        "checkpoint_source": "hugging_face",
        "hf_repo": "neurovlm/3d_cnn",
        "hf_filename": "mixed_ae.pt",
    }]


load_stage1a_enabled = bool(LOAD_STAGE1A.get("enabled", False))
load_stage1a_source = str(LOAD_STAGE1A.get("source", "hf")).lower()
if load_stage1a_enabled:
    if load_stage1a_source == "hf":
        results = load_stage1a_result_from_hf()
        stage1a_loaded_existing_count = len(results)
        stage1a_status = "loaded from Hugging Face checkpoint"
        run_recipes = []
    elif load_stage1a_source == "run":
        source_run_dir = str(LOAD_STAGE1A.get("run_dir", ""))
        if not source_run_dir:
            raise ValueError("Set LOAD_STAGE1A['run_dir'] or NEUROVLM_STAGE1A_SOURCE_RUN_DIR when LOAD_STAGE1A loads from a run")
        results = load_stage1a_results_from_run(source_run_dir)
        stage1a_loaded_existing_count = len(results)
        stage1a_status = "loaded from existing checkpoints"
        run_recipes = []
    else:
        raise ValueError("LOAD_STAGE1A['source'] must be 'hf' or 'run'")
elif not RUN_STAGE1A_MIXED_PRETRAINING:
    print("Stage 1A mixed pretraining skipped intentionally.")
    stage1a_status = "skipped intentionally"
    run_recipes = []
else:
    stage1a_status = "run"

for recipe in run_recipes:
    out_dir = RUN_DIR / "01_stage1_ae_pretraining" / recipe["ae_variant"]
    ckpt_dir = out_dir / "checkpoints"
    sel_metric = recipe.get("checkpoint_selection_metric", "best_val_loss")
    canonical_name = AE_SELECTION_TO_FILE.get(sel_metric, "best_val_loss.pt")
    best_checkpoint = ckpt_dir / canonical_name
    done_marker = out_dir / "training_stop.json"
    if RESUME_COMPLETED and best_checkpoint.exists() and done_marker.exists():
        print(f"Skipping completed AE variant {recipe['ae_variant']}: {best_checkpoint}")
        ensure_checkpoint_eval_table(out_dir)
        results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": str(ckpt_dir), "best_checkpoint": str(best_checkpoint)})
        stage1a_loaded_existing_count += 1
        continue

    print(f"Running AE variant: {recipe['ae_variant']} ({AE_EPOCHS} epochs, mode={AE_ABLATION_MODE})")
    base_cfg = {
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(out_dir),
        "checkpoint_dir": str(out_dir / "checkpoints"),
        "data_mode": "mixed",
        "target_shape": [36, 45, 38],
        "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "amp": True,
        "gradient_clipping": 1.0,
        "batch_size": 64,
        "epochs": AE_EPOCHS,
        "early_stopping_patience": AE_EARLY_STOPPING_PATIENCE,
        "max_train_batches": MAX_TRAIN_BATCHES,
        "max_val_batches": MAX_VAL_BATCHES,
        "train_metric_batches": TRAIN_METRIC_BATCHES,
        "val_metric_batches": VAL_METRIC_BATCHES,
        "final_eval": FINAL_EVAL,
        "save_plots": SAVE_PLOTS,
        "progress": True,
        "num_workers": NUM_WORKERS,
        "eval_num_workers": EVAL_NUM_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": True,
        "persistent_workers": NUM_WORKERS > 0,
        "metrics_device": METRICS_DEVICE,
        "compute_epoch_source_metrics": COMPUTE_EPOCH_SOURCE_METRICS,
        "compute_train_metrics": COMPUTE_TRAIN_METRICS,
        "save_legacy_ae_checkpoint_aliases": SAVE_LEGACY_AE_CHECKPOINT_ALIASES,
        **recipe,
    }
    result = train_with_batch_fallback(base_cfg, AE_BATCH_CANDIDATES)
    results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": result["checkpoint_dir"], "best_checkpoint": result["best_checkpoint"]})
    stage1a_trained_count += 1
    clear_cuda_memory()
if stage1a_status == "run" and stage1a_trained_count == 0 and stage1a_loaded_existing_count:
    stage1a_status = "loaded from existing checkpoints"
elif stage1a_status == "run" and stage1a_trained_count and stage1a_loaded_existing_count:
    stage1a_status = "run; loaded existing checkpoints"
elif stage1a_status == "run" and not results:
    stage1a_status = "missing"
print("Stage 1A status:", stage1a_status)
results


In [ ]:
stage1b_status = "skipped intentionally"
stage1b_trained_domains = []
stage1b_loaded_domains = []
stage1b_missing_reason = ""

if RUN_STAGE1B_FINETUNING and results:
    seed_matches = [row for row in results if row.get("ae_variant") == STAGE1B_SEED_VARIANT]
    if not seed_matches:
        available = [row.get("ae_variant") for row in results]
        raise ValueError(f"STAGE1B_SEED_VARIANT={STAGE1B_SEED_VARIANT!r} not found in results. Available: {available}")
    seed_checkpoint = seed_matches[-1]["best_checkpoint"]
    print("Stage 1B seed variant:", STAGE1B_SEED_VARIANT)
    print("Stage 1B seed checkpoint:", seed_checkpoint)
    for mode, domain, ae_variant in STAGE1B_DOMAINS:
        out_dir = RUN_DIR / "02_stage1b_ae_finetuning" / domain
        best_checkpoint = out_dir / "checkpoints" / "best_cnn_autoencoder.pt"
        done_marker = out_dir / "training_stop.json"
        if RESUME_COMPLETED and best_checkpoint.exists() and done_marker.exists():
            print(f"Loading existing completed Stage 1B {domain}: {best_checkpoint}")
            ensure_checkpoint_eval_table(out_dir)
            results.append({"ae_variant": ae_variant, "checkpoint_dir": str(out_dir / "checkpoints"), "best_checkpoint": str(best_checkpoint), "seed_variant": STAGE1B_SEED_VARIANT, "seed_checkpoint": seed_checkpoint})
            stage1b_loaded_domains.append(domain)
            continue
        cfg = {
            "stage1b_mode": mode,
            "mixed_pretrain_checkpoint": seed_checkpoint,
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "output_dir": str(out_dir),
            "checkpoint_dir": str(out_dir / "checkpoints"),
            "ae_variant": ae_variant,
            "stage1b_seed_variant": STAGE1B_SEED_VARIANT,
            "stage1b_seed_checkpoint": seed_checkpoint,
            "lr": 1e-4,
            "freeze_mode": "none",
            "epochs": STAGE1B_EPOCHS,
            "early_stopping_patience": AE_EARLY_STOPPING_PATIENCE,
            "batch_size": min(64, max(STAGE1B_BATCH_CANDIDATES)),
            "batch_candidates": STAGE1B_BATCH_CANDIDATES,
            "max_batch_size": max(STAGE1B_BATCH_CANDIDATES),
            "preflight_vram_reserve_gb": STAGE1B_PREFLIGHT_RESERVE_GB,
            "max_train_batches": MAX_TRAIN_BATCHES,
            "max_val_batches": MAX_VAL_BATCHES,
            "train_metric_batches": TRAIN_METRIC_BATCHES,
            "val_metric_batches": VAL_METRIC_BATCHES,
            "final_eval": FINAL_EVAL,
            "save_plots": SAVE_PLOTS,
            "loss": {"type": "raw_mse", "prediction_activation": "none"},
            "num_workers": NUM_WORKERS,
            "eval_num_workers": EVAL_NUM_WORKERS,
            "prefetch_factor": PREFETCH_FACTOR,
            "pin_memory": True,
            "persistent_workers": NUM_WORKERS > 0,
            "metrics_device": METRICS_DEVICE,
            "compute_epoch_source_metrics": COMPUTE_EPOCH_SOURCE_METRICS,
            "compute_train_metrics": COMPUTE_TRAIN_METRICS,
        }
        ft_result = train_with_batch_fallback(cfg, STAGE1B_BATCH_CANDIDATES, reserve_gb=STAGE1B_PREFLIGHT_RESERVE_GB, train_fn=train_stage1b_from_config)
        results.append({"ae_variant": cfg["ae_variant"], "checkpoint_dir": ft_result["checkpoint_dir"], "best_checkpoint": ft_result["best_checkpoint"], "seed_variant": STAGE1B_SEED_VARIANT, "seed_checkpoint": seed_checkpoint})
        stage1b_trained_domains.append(domain)
        clear_cuda_memory()
    if stage1b_trained_domains and stage1b_loaded_domains:
        stage1b_status = "run; loaded from existing checkpoints"
    elif stage1b_trained_domains:
        stage1b_status = "run"
    elif stage1b_loaded_domains:
        stage1b_status = "loaded from existing checkpoints"
elif RUN_STAGE1B_FINETUNING:
    stage1b_status = "missing"
    stage1b_missing_reason = "Stage 1B requested, but no Stage 1 results are available."
    print(stage1b_missing_reason)
else:
    stage1b_status = "skipped intentionally"
    print("Stage 1B fine-tuning skipped intentionally (RUN_STAGE1B_FINETUNING=False).")

print("Stage 1B status:", stage1b_status)


In [ ]:
stage_control_status = {
    "RUN_STAGE1A_MIXED_PRETRAINING": RUN_STAGE1A_MIXED_PRETRAINING,
    "AE_EARLY_STOPPING_PATIENCE": AE_EARLY_STOPPING_PATIENCE,
    "TRAIN_OTHER_AE_VARIANTS": TRAIN_OTHER_AE_VARIANTS,
    "SELECTED_AE_VARIANTS": SELECTED_AE_VARIANTS,
    "LOAD_STAGE1A": LOAD_STAGE1A,
    "STAGE1A_HF_CHECKPOINT": "neurovlm/3d_cnn/mixed_ae.pt",
    "RUN_STAGE1B_FINETUNING": RUN_STAGE1B_FINETUNING,
    "stage1a_status": globals().get("stage1a_status", "missing"),
    "stage1a_trained_count": globals().get("stage1a_trained_count", 0),
    "stage1a_loaded_existing_count": globals().get("stage1a_loaded_existing_count", 0),
    "stage1b_status": globals().get("stage1b_status", "missing"),
    "stage1b_trained_domains": globals().get("stage1b_trained_domains", []),
    "stage1b_loaded_domains": globals().get("stage1b_loaded_domains", []),
    "stage1b_missing_reason": globals().get("stage1b_missing_reason", ""),
}
metadata_dir = RUN_DIR / "00_run_metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)
(metadata_dir / "stage1_stage1b_control_status.json").write_text(json.dumps(stage_control_status, indent=2))

summary_rows = []
for row in results:
    run_dir = Path(row["checkpoint_dir"]).parent
    checkpoint_metrics_path = run_dir / "metrics" / "reconstruction_summary_by_checkpoint_source.csv"
    source_metrics_path = run_dir / "metrics" / "reconstruction_summary_by_source.csv"
    metrics_path = checkpoint_metrics_path if checkpoint_metrics_path.exists() else source_metrics_path
    grouped = {}
    if metrics_path.exists():
        with metrics_path.open(newline="") as f:
            for m in csv.DictReader(f):
                if m.get("split") not in {"val", "test"}:
                    continue
                if m.get("source_detail") != "ALL_DETAILS" or m.get("source") not in {"pubmed", "neurovault", "nilearn"}:
                    continue
                checkpoint = m.get("checkpoint", "last")
                split = m.get("split", "")
                key = (checkpoint, split)
                item = grouped.setdefault(key, {**row, "checkpoint": checkpoint, "split": split, "metrics_path": str(metrics_path)})
                if m.get("checkpoint_path"):
                    item["evaluated_checkpoint_path"] = m.get("checkpoint_path")
                src = m["source"]
                item[f"{src}_spatial_corr"] = m.get("spatial_corr", "")
                item[f"{src}_top5_dice"] = m.get("top5_dice", "")
                item[f"{src}_mse"] = m.get("mse", m.get("reconstruction_mse", ""))
    for item in grouped.values():
        values = []
        for key, value in item.items():
            if key.endswith("_spatial_corr") or key.endswith("_top5_dice"):
                try:
                    values.append(float(value))
                except Exception:
                    pass
        item["ranking_score"] = sum(values) / len(values) if values else ""
        item["recommendation_note"] = "Prioritize spatial_corr/top5_dice by source; do not rank by MSE alone."
        summary_rows.append(item)
summary_rows = sorted(summary_rows, key=lambda r: float(r["ranking_score"]) if r["ranking_score"] != "" else -999, reverse=True)
final_dir = RUN_DIR / "07_final_comparison"
final_dir.mkdir(parents=True, exist_ok=True)
readme = f"""# Stage 1 AE Ablation Status

RUN_STAGE1A_MIXED_PRETRAINING = {RUN_STAGE1A_MIXED_PRETRAINING}
AE_EARLY_STOPPING_PATIENCE = {AE_EARLY_STOPPING_PATIENCE}
TRAIN_OTHER_AE_VARIANTS = {TRAIN_OTHER_AE_VARIANTS}
SELECTED_AE_VARIANTS = {SELECTED_AE_VARIANTS}
LOAD_STAGE1A = {LOAD_STAGE1A}
STAGE1A_HF_CHECKPOINT = neurovlm/3d_cnn/mixed_ae.pt
RUN_STAGE1B_FINETUNING = {RUN_STAGE1B_FINETUNING}

Stage 1A status: {stage_control_status['stage1a_status']}
Stage 1B status: {stage_control_status['stage1b_status']}
Stage 1B trained domains: {', '.join(stage_control_status['stage1b_trained_domains']) if stage_control_status['stage1b_trained_domains'] else 'none'}
Stage 1B loaded from existing checkpoints: {', '.join(stage_control_status['stage1b_loaded_domains']) if stage_control_status['stage1b_loaded_domains'] else 'none'}
Stage 1B missing reason: {stage_control_status['stage1b_missing_reason'] or 'none'}
"""
(final_dir / "README_WHAT_TO_LOOK_AT.md").write_text(readme)
write_table(final_dir / "stage1_stage1b_control_status.csv", [stage_control_status])
write_table(RUN_DIR / "07_final_comparison" / "best_checkpoints_to_inspect.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "final_summary_table.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "ae_ablation_leaderboard.csv", summary_rows)
print("AE ablation summary written to", RUN_DIR / "07_final_comparison")
summary_rows
